In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load dataset
df = pd.read_csv("AQI_dataset.csv")

# Aggregate pollutant data by city (mean values per city)
df_grouped = df.groupby("city")["pollutant_avg"].mean().reset_index()
df_grouped.rename(columns={"pollutant_avg": "AQI"}, inplace=True)

# Select relevant pollutants for AQI prediction
pollutants = df.pivot_table(index=["city"], columns="pollutant_id", values="pollutant_avg", aggfunc="mean")
df_merged = df_grouped.merge(pollutants, on="city", how="left").dropna()

# Define features & target
features = list(pollutants.columns)
target = "AQI"

X = df_merged[features]
y = df_merged[target]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Linear Regression model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Save model & scaler
pickle.dump(model, open("aqi_model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))

print("✅ Model training complete! Saved as 'aqi_model.pkl'.")


✅ Model training complete! Saved as 'aqi_model.pkl'.


In [2]:
print("Features used for training:", X.columns.tolist())


Features used for training: ['CO', 'NH3', 'NO2', 'OZONE', 'PM10', 'PM2.5', 'SO2']
